<a href="https://colab.research.google.com/github/jsalafica/Data-Science-III/blob/master/Entrega_Final_DSIII_Javier_Salafica3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrega Final — NLP + Deep Learning (Clasificación de texto clínico)

**Objetivo:** construir un pipeline de NLP en español para clasificar textos clínicos simples en un único servicio médico (single-label), aplicando preprocesamiento y entrenando:
1) un modelo clásico (Regresión Logística)
2) una red neuronal simple (Deep Learning) con Keras

**Dataset:** CSV con ≥1000 filas, cargado desde GitHub.


In [1]:
!pip -q install spacy
!python -m spacy download es_core_news_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 46.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
!pip -q install wordcloud
from wordcloud import WordCloud

In [3]:
import re
import numpy as np
import pandas as pd

import spacy
from spacy.lang.es.stop_words import STOP_WORDS

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from tensorflow import keras
from tensorflow.keras import layers


## 1. Carga del dataset

Se carga el dataset desde GitHub (formato CSV, separador `;`).  
El dataset contiene las columnas:

- `texto`: texto clínico en español  
- `servicio`: etiqueta (single-label)


In [ ]:
URL = "https://raw.githubusercontent.com/jsalafica/Data-Science-III/refs/heads/master/dataset_nlp_1200.csv"

df = pd.read_csv(URL, sep=";")

print(df.shape)
df.head()

In [ ]:
print(df.columns)
print(df["servicio"].value_counts().head(10))
print("N clases:", df["servicio"].nunique())
print("N nulos texto:", df["texto"].isna().sum())
print("N nulos servicio:", df["servicio"].isna().sum())


## 2. Preprocesamiento NLP

Se aplica preprocesamiento con **spaCy**:

- Normalización (minúsculas y limpieza básica)
- Tokenización y lematización
- Eliminación de stopwords
- Preservación de información etaria mediante tokens semánticos:
  - `edad_adulto`
  - `edad_pediatrico`
  - `edad_neonatal`


In [6]:
nlp = spacy.load("es_core_news_sm")
stop_es = set(STOP_WORDS)

len(stop_es)


521

In [7]:
def preprocess_spacy(texto: str) -> str:
    texto = str(texto).lower()
    texto = re.sub(r"\s+", " ", texto).strip()

    # ---- Preservación etaria CORRECTA ----
    pat_edad = re.compile(r"\b(\d{1,3})\s*(años?|mes(?:es)?|d[ií]as?|horas?)\b", flags=re.IGNORECASE)

    def _edad_a_dias(valor: int, unidad: str) -> int:
        unidad = unidad.lower()
        if unidad.startswith("año"):
            return int(valor * 365)
        if unidad.startswith("mes"):
            return int(valor * 30)
        if unidad.startswith("día") or unidad.startswith("dia"):
            return int(valor)
        if unidad.startswith("hora"):
            return 0
        return None

    def _clasificar_edad_por_dias(dias: int) -> str:
        if dias is not None and dias < 30:
            return "edad_neonatal"
        if dias is not None and dias < (16 * 365):
            return "edad_pediatrico"
        return "edad_adulto"

    def _reemplazar_edad(match):
        valor = int(match.group(1))
        unidad = match.group(2)
        dias = _edad_a_dias(valor, unidad)
        token = _clasificar_edad_por_dias(dias)
        return f" {token} "

    texto = pat_edad.sub(_reemplazar_edad, texto)

    # ---- spaCy ----
    doc = nlp(texto)

    tokens = []
    for tok in doc:
        if tok.is_space or tok.is_punct:
            continue

        lemma = tok.lemma_.strip()
        if not lemma:
            continue

        # eliminar números puros (edad ya fue tokenizada)
        if lemma.isdigit():
            continue

        if lemma in stop_es:
            continue

        tokens.append(lemma)

    return " ".join(tokens)


In [ ]:
df["texto_proc"] = df["texto"].astype(str).apply(preprocess_spacy)

df[["texto", "texto_proc"]].head(10)


### Nube de palabras

Se generó una nube de palabras a partir del texto preprocesado, lo cual permite
visualizar los términos más frecuentes del dataset y obtener una primera
aproximación exploratoria al contenido textual.


In [ ]:
texto_total = " ".join(df["texto_proc"].dropna().astype(str).tolist())

wc = WordCloud(
    width=1400,
    height=800,
    background_color="white",
    collocations=False  # evita juntar bigramas raros
).generate(texto_total)

plt.figure(figsize=(14, 8))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Mapa de palabras (dataset completo)")
plt.show()


In [ ]:
def wordcloud_por_servicio(servicio, max_words=200):
    subset = df.loc[df["servicio"] == servicio, "texto_proc"].dropna().astype(str)
    texto = " ".join(subset.tolist())

    wc = WordCloud(
        width=1400,
        height=800,
        background_color="white",
        max_words=max_words,
        collocations=False
    ).generate(texto)

    plt.figure(figsize=(14, 8))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Mapa de palabras — {servicio}")
    plt.show()

# Ejemplo:
wordcloud_por_servicio("UTI Adultos")


## 3. Vectorización (TF-IDF)

Transformamos el texto preprocesado a una matriz numérica usando TF-IDF.
Se utilizan unigramas y bigramas.


In [11]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=8000
)

X = vectorizer.fit_transform(df["texto_proc"])
y = df["servicio"]

print("X shape:", X.shape)
print("Clases:", y.nunique())


X shape: (1200, 929)
Clases: 16


## 4. Modelo clásico (Regresión Logística)

Se entrena un modelo base de clasificación multiclase usando Regresión Logística.
Se divide el dataset en train/test 80/20 con estratificación por clase.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


## 5. Análisis de resultados (Profundización NLP)

Se analiza la matriz de confusión para observar clases que se confunden entre sí.


In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(12, 12))
disp.plot(ax=ax, xticks_rotation=90, values_format=".2f", colorbar=True)
plt.title("Matriz de confusión normalizada (por clase real)")
plt.tight_layout()
plt.show()


In [14]:
cm_raw = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm_raw, index=labels, columns=labels)

cm_err = cm_df.copy()
np.fill_diagonal(cm_err.values, 0)

top = cm_err.stack().sort_values(ascending=False).head(15)

print("Top 15 confusiones (Real -> Predicho : cantidad):")
for (real, pred), cnt in top.items():
    if cnt > 0:
        print(f"{real} -> {pred}: {cnt}")


Top 15 confusiones (Real -> Predicho : cantidad):


## 6. Interpretabilidad del modelo

La Regresión Logística permite inspeccionar los términos con mayor peso por clase.
Además, se muestra una explicación simple de una predicción individual.


In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
classes = model.classes_
coef = model.coef_

def top_terms_for_class(class_idx, top_n=12):
    top_pos_idx = np.argsort(coef[class_idx])[-top_n:][::-1]
    return list(zip(feature_names[top_pos_idx], coef[class_idx][top_pos_idx]))

for i, cls in enumerate(classes):
    tops = top_terms_for_class(i, top_n=10)
    print(f"\n=== {cls} | Top términos ===")
    for term, w in tops:
        print(f"{term:25s} {w:.3f}")


In [42]:
def explain_prediction(texto_original, top_k=12):
    texto_p = preprocess_spacy(texto_original)
    X_one = vectorizer.transform([texto_p])

    scores = model.decision_function(X_one).ravel()
    pred_idx = np.argmax(scores)
    pred_class = model.classes_[pred_idx]

    row = X_one.tocoo()
    contrib = {}
    for j, v in zip(row.col, row.data):
        contrib[j] = contrib.get(j, 0.0) + (coef[pred_idx, j] * v)

    top = sorted(contrib.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
    top_terms = [(feature_names[j], float(val)) for j, val in top]

    return {"texto_proc": texto_p, "prediccion": pred_class, "top_terms": top_terms}

#ejemplo = "Niño 3 años con dificultad respiratoria severa, tiraje y saturación baja."
##ejemplo = "Paciente primigesta 29.1 semanas con FUM 7/7 con ecografia acorde a 16.3 semanas. antecedente de epilepsia en tratamiento con acido valproico. consulta por dolor abdominal, normotensa, afebril, se constata al ingreso 3 contracciones de 30 segundos en 10 minutos, TV posterior 2 de largo oce al pulpejo. Se indica nifedipina que responde parcialmente. Se realiza nuevamente TV centralizado 70%borrado, oce al pulpejo y se suspende nifedipina. Labo con GB 10250, orina con Trichomonas, se interpreta el cuadro como corioamnionitis, se indica ampicilina gentamicina y betametasona para maduracion pulmonar. Presento como complicacion una convulsion tomica-clonica con recuperacion adintegrum iniciándose levetiracetam. Se solicita derivacion en codigo rojo a efector correspondiente por edad gestacional."
ejemplo = "Paciente de 46 años con antecedente mencionados consulta por cuadro de 2 semanas de evolución caracterizado por cefalea holocraneana de intensidad 7/10 sin irradiación que se asocia a sensación febril y dos episodios de vómitos de contenido gástrico no precedidos por nauseas. Al interrogatorio dirigido refiere presentar de 1 mes de evolución, diarrea acuosa no disenteriforme, tos seca y disfagia. Ingresa a Clínica médica para diagnóstico y tratamiento"
info = explain_prediction(ejemplo, top_k=12)

print("Texto procesado:", info["texto_proc"])
print("Predicción:", info["prediccion"])
print("Top términos que empujaron la predicción:")
for t, v in info["top_terms"]:
    print(f"{t:25s} {v:.4f}")


Texto procesado: paciente edad_adulto antecedente mencionado consulta cuadro semana evolución caracterizado cefalea holocraneán intensidad 7/10 irradiación asociar sensación febril episodio vómito contenido gástrico precedido nausea interrogatorio dirigido referir presentar edad_pediatrico evolución diarrea acuós disenterifor yo to seco disfagia ingresar clínica médico diagnóstico tratamiento
Predicción: Clinica Medica
Top términos que empujaron la predicción:
seco                      0.3295
to seco                   0.3295
cefalea                   0.1859
diarrea                   0.1765
to                        0.1734
paciente                  0.0959
vómito                    0.0934
edad_adulto               0.0104
antecedente               -0.0260
sensación                 -0.0317
semana                    -0.0501
edad_pediatrico           -0.0961


In [47]:
def recomendar_servicios(texto, top_k=5, threshold=0.15):
    t = preprocess_spacy(texto)
    v = vectorizer.transform([t])
    probs = model.predict_proba(v)[0]
    pares = sorted(zip(model.classes_, probs), key=lambda x: x[1], reverse=True)

    top = [(c, float(p)) for c, p in pares[:top_k]]
    por_umbral = [(c, float(p)) for c, p in pares if p >= threshold]

    return {"top_k": top, "por_umbral": por_umbral}

recomendar_servicios("Paciente de 46 años con antecedente mencionados consulta por cuadro de 2 semanas de evolución caracterizado por cefalea holocraneana de intensidad 7/10 sin irradiación que se asocia a sensación febril y dos episodios de vómitos de contenido gástrico no precedidos por nauseas. Al interrogatorio dirigido refiere presentar de 1 mes de evolución, diarrea acuosa no disenteriforme, tos seca y disfagia. Ingresa a Clínica médica para diagnóstico y tratamiento")


{'top_k': [('Clinica Medica', 0.19487320704341252),
  ('Pediatria', 0.1791693358574388),
  ('Obstetricia', 0.08884183995455719),
  ('Neurocirugia', 0.055578286720318844),
  ('UTI Pediatrica', 0.05296095454074744)],
 'por_umbral': [('Clinica Medica', 0.19487320704341252),
  ('Pediatria', 0.1791693358574388)]}

## 7. Deep Learning (Red neuronal simple)

Se entrena una red neuronal feed-forward sencilla usando como entrada la matriz TF-IDF.


In [17]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

X_train_d = X_train.toarray()
X_test_d  = X_test.toarray()

y_train_oh = keras.utils.to_categorical(y_train, num_classes=num_classes)
y_test_oh  = keras.utils.to_categorical(y_test,  num_classes=num_classes)

X_train_d.shape, y_train_oh.shape, num_classes


((960, 929), (960, 16), 16)

In [18]:
input_dim = X_train_d.shape[1]

model_nn = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model_nn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_nn.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       119,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         2,064 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,104 (473.06 KB)

 Trainable params: 121,104 (473.06 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
early = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model_nn.fit(
    X_train_d, y_train_oh,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    callbacks=[early],
    verbose=1
)


Epoch 1/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.3271 - loss: 2.7104 - val_accuracy: 0.9844 - val_loss: 2.5044
Epoch 2/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9799 - loss: 2.4008 - val_accuracy: 1.0000 - val_loss: 2.0863
Epoch 3/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 1.9161 - val_accuracy: 1.0000 - val_loss: 1.4667
Epoch 4/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 1.2662 - val_accuracy: 1.0000 - val_loss: 0.8307
Epoch 5/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.6755 - val_accuracy: 1.0000 - val_loss: 0.4161
Epoch 6/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.3389 - val_accuracy: 1.0000 - val_loss: 0.2207
Epoch 7/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.1893 - val_accuracy: 1.0000 - val_loss: 0.1337
Epoch 8/15
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.1146 - val_accuracy: 1.0000 - val_loss

In [20]:
y_pred_prob = model_nn.predict(X_test_d)
y_pred = np.argmax(y_pred_prob, axis=1)

print("Accuracy DL:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
Accuracy DL: 1.0
                      precision    recall  f1-score   support

         Cardiologia       1.00      1.00      1.00        15
     Cirugia General       1.00      1.00      1.00        15
      Clinica Medica       1.00      1.00      1.00        15
   Gastroenterologia       1.00      1.00      1.00        15
         Ginecologia       1.00      1.00      1.00        15
        Neonatologia       1.00      1.00      1.00        15
        Neurocirugia       1.00      1.00      1.00        15
          Neurologia       1.00      1.00      1.00        15
         Obstetricia       1.00      1.00      1.00        15
Otorrinolaringologia       1.00      1.00      1.00        15
           Pediatria       1.00      1.00      1.00        15
       Traumatologia       1.00      1.00      1.00        15
         UTI Adultos       1.00      1.00      1.00        15
        UTI Neonatal       1.00      1.00      1.00        15
      UTI Pedi

In [40]:
def predict_nn(texto: str, top_k=5):
    t = preprocess_spacy(texto)
    v = vectorizer.transform([t]).toarray()
    p = model_nn.predict(v)[0]
    idx = np.argsort(p)[::-1][:top_k]
    return [(le.classes_[i], float(p[i])) for i in idx]

##predict_nn("Paciente primigesta 29.1 semanas con FUM 7/7 con ecografia acorde a 16.3 semanas. antecedente de epilepsia en tratamiento con acido valproico. consulta por dolor abdominal, normotensa, afebril, se constata al ingreso 3 contracciones de 30 segundos en 10 minutos, TV posterior 2 de largo oce al pulpejo. Se indica nifedipina que responde parcialmente. Se realiza nuevamente TV centralizado 70%borrado, oce al pulpejo y se suspende nifedipina. Labo con GB 10250, orina con Trichomonas, se interpreta el cuadro como corioamnionitis, se indica ampicilina gentamicina y betametasona para maduracion pulmonar. Presento como complicacion una convulsion tomica-clonica con recuperacion adintegrum iniciándose levetiracetam. Se solicita derivacion en codigo rojo a efector correspondiente por edad gestacional.", top_k=5)
##predict_nn("Paciente de 46 años con antecedente mencionados consulta por cuadro de 2 semanas de evolución caracterizado por cefalea holocraneana de intensidad 7/10 sin irradiación que se asocia a sensación febril y dos episodios de vómitos de contenido gástrico no precedidos por nauseas. Al interrogatorio dirigido refiere presentar de 1 mes de evolución, diarrea acuosa no disenteriforme, tos seca y disfagia. Ingresa a Clínica médica para diagnóstico y tratamiento", top_k=5)
##predict_nn("Varón de 58 años con antecedente de hipertensión arterial consulta por dolor precordial opresivo de inicio súbito irradiado a brazo izquierdo, acompañado de sudoración fría y náuseas. Se constata taquicardia y se decide internación para evaluación cardiológica.")
predict_nn("Paciente de 72 años con antecedente de EPOC y cardiopatía refiere disnea progresiva, tos productiva y febrícula de varios días de evolución. Presenta decaimiento general y saturación de oxígeno de 85% con requerimiento de oxigeno requiere ventilación mecánica invasiva")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step


[('Clinica Medica', 0.5736986398696899),
 ('Ginecologia', 0.04781225696206093),
 ('UTI Neonatal', 0.041109081357717514),
 ('Neonatologia', 0.037010010331869125),
 ('Gastroenterologia', 0.0340905524790287)]

## 8. Conclusiones

- Se construyó un pipeline de NLP en español con spaCy (tokenización, lematización, stopwords) y preservación de edad mediante tokens semánticos.
- Se vectorizó el texto con TF-IDF y se entrenó un modelo clásico (Regresión Logística) con evaluación mediante accuracy, reporte de clasificación y matriz de confusión.
- Se implementó una red neuronal simple (Deep Learning) usando Keras, cumpliendo el requisito mínimo solicitado.

**Trabajo futuro:** evaluar arquitecturas más potentes (Embeddings + LSTM/Transformers) y ampliar variabilidad de textos.
